# Copy n=3205 1000genomesVDS to TRUST staging 

add sample_info based on IGSR sample info 
- biosampleID - > `pid`
- sample_name -> `npmid` (& `s`)

In [1]:
%%configure -f
{
    "driverMemory": "45G"
}

In [2]:
# Import and initiate HAIL
import hail as hl
hl.init(sc,log='/tmp/hail.log')

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1755679836912_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.5.2-amzn-1
SparkUI available at http://ip-192-168-77-183.ap-southeast-1.compute.internal:36299
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.134-952ae203dbbe
LOGGING: writing to /tmp/hail.log

In [3]:
from pprint import pprint
pprint(dict(hl.spark_context().getConf().getAll()))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

{'spark.app.attempt.id': '1',
 'spark.app.id': 'application_1755679836912_0001',
 'spark.app.name': 'livy-session-0',
 'spark.app.startTime': '1755680616198',
 'spark.app.submitTime': '1755680596559',
 'spark.blacklist.decommissioning.enabled': 'true',
 'spark.blacklist.decommissioning.timeout': '1h',
 'spark.decommissioning.timeout.threshold': '20',
 'spark.default.parallelism': '1024',
 'spark.driver.defaultJavaOptions': "-XX:OnOutOfMemoryError='kill -9 %p'",
 'spark.driver.extraClassPath': '/usr/local/lib/python3.9/site-packages/hail/backend/hail-all-spark.jar:/usr/lib/hadoop-lzo/lib/*:/usr/lib/hadoop/hadoop-aws.jar:/usr/share/aws/aws-java-sdk/*:/usr/share/aws/emr/emrfs/conf:/usr/share/aws/emr/emrfs/lib/*:/usr/share/aws/emr/emrfs/auxlib/*:/usr/share/aws/emr/goodies/lib/emr-spark-goodies.jar:/usr/share/aws/emr/security/conf:/usr/share/aws/emr/security/lib/*:/usr/share/aws/hmclient/lib/aws-glue-datacatalog-spark-client.jar:/usr/share/java/Hive-JSON-Serde/hive-openx-serde.jar:/usr/shar

In [46]:
# source

vds_prefix = 's3://precise-scratch/goypav/1KG/'
trust_prefix = 's3://precise-trust/opendata-1000genomes/'


# input
vds_uri = vds_prefix + 'VDS/1000genomes_combined_batch1_2_3_4.bf2-tr500k-sp1k.n3205.vds'
tsv_uri = vds_prefix +  'igsr_sample_info.tsv'

# output
vds_trust_uri = trust_prefix + '1000genomes-vds-n3205.vds'

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [106]:
## parse igsr_sample_info

info_ht = hl.import_table(tsv_uri)
info_ht = info_ht.rename({
    'Sample name': 'npmid', 
    'Sex': 'sex',
    'Biosample ID': 'pid',
    'Population code': 'population_code', 
    'Population name': 'population_name', 
    'Superpopulation code': 'superpopulation_code', 
    'Superpopulation name': 'superpopulation_name', 
    'Population elastic ID': 'population_elastic_id', 
    'Data collections': 'data_collections'
})

## GiAB samples (HG00{2,3,4}) not defined in igsr_info !!!!
# build extra rows (only set the needed fields)
extra = hl.Table.parallelize([
    {'npmid': 'HG002', 'pid': 'GIAB_sample_HG002', 'data_collections': 'The Genome in a Bottle (GIAB) Consortium'},
    {'npmid': 'HG003', 'pid': 'GIAB_sample_HG003', 'data_collections': 'The Genome in a Bottle (GIAB) Consortium'},
    {'npmid': 'HG004', 'pid': 'GIAB_sample_HG004', 'data_collections': 'The Genome in a Bottle (GIAB) Consortium'},
])

# align types/columns to match info_ht
row_t = info_ht.row.dtype  # dict-like: field -> HailType
for fname, ftype in row_t.items():
    if fname not in extra.row:                 # add any missing columns as nulls
        extra = extra.annotate(**{fname: hl.null(ftype)})
extra = extra.select(*row_t.keys()) # re-order

# union
info_ht = info_ht.union(extra)

# re-arrange
info_ht = info_ht.transmute(data_collections  = info_ht.data_collections.split(','))
info_ht = info_ht.drop('population_elastic_id')
info_ht = info_ht.annotate(s = info_ht.npmid)
info_ht = info_ht.key_by('s')

info_ht = info_ht.annotate(sample_info = hl.struct())
info_ht = info_ht.annotate(sample_info = info_ht.sample_info.annotate(**info_ht[info_ht.key])).select('sample_info')
info_ht = info_ht.annotate(sample_info = info_ht.sample_info.drop('sample_info'))

info_ht.describe()
info_ht.count() ## 4981 igsr_info also contains Simons Genome Diversity and few other none 1000genomes samples 

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    's': str 
    'sample_info': struct {
        npmid: str, 
        sex: str, 
        pid: str, 
        population_code: str, 
        population_name: str, 
        superpopulation_code: str, 
        superpopulation_name: str, 
        data_collections: array<str>
    } 
----------------------------------------
Key: ['s']
----------------------------------------
4981
2025-08-20 12:26:33.043 Hail: INFO: Reading table without type imputation
  Loading field 'Sample name' as type str (not specified)
  Loading field 'Sex' as type str (not specified)
  Loading field 'Biosample ID' as type str (not specified)
  Loading field 'Population code' as type str (not specified)
  Loading field 'Population name' as type str (not specified)
  Loading field 'Superpopulation code' as type str (not specified)
  Loading field 'Superpopulation name' as type str (not specified)
  Load

In [48]:
vds = hl.vds.read_vds(vds_uri)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [49]:
vds.reference_data.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    'ref_block_max_length': int32
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
----------------------------------------
Entry fields:
    'LEN': int32
    'DP': int32
    'GQ': int32
    'ICNT': array<int32>
    'MIN_DP': int32
    'SPL': array<int32>
    'LGT': call
    'LAD': array<int32>
    'END': int32
----------------------------------------
Column key: ['s']
Row key: ['locus']
----------------------------------------

In [50]:
hl.eval(vds.reference_data.ref_block_max_length)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

151393

In [51]:
vds.variant_data.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
----------------------------------------
Entry fields:
    'LA': array<int32>
    'LGT': call
    'LAD': array<int32>
    'LPL': array<int32>
    'RGQ': int32
    'gvcf_info': struct {
        DB: bool, 
        FS: float64, 
        FractionInformativeReads: float64, 
        LOD: float64, 
        MQ: float64, 
        MQRankSum: float64, 
        QD: float64, 
        R2_5P_bias: float64, 
        ReadPosRankSum: float64, 
        SOR: float64
    }
    'AF': array<float64>
    'DP': int32
    'F1R2': array<int32>
    'F2R1': array<int32>
    'GP': array<float64>
    'GQ': int32
    'ICNT': array<int32>
    'MB': array<int32>
    'MIN_DP': int32
    'PRI': array<float64>
    'PS': int32
    'SB': array<int32>
    'SPL': array<int32

In [52]:
# Count rows/cols in the variant_data MT
print(f"Reference genome: {vds.variant_data.locus.dtype.reference_genome.name}")
print(f"Number of samples: {vds.n_samples()}")
print(f"Number of variant partitions: {vds.variant_data.n_partitions()}")
print(f"Total number of variants: {vds.variant_data.count_rows():,}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Reference genome: GRCh38
Number of samples: 3205
Number of variant partitions: 5767
Total number of variants: 156,228,032

In [104]:
vd = vds.variant_data
vd = vd.annotate_cols(**info_ht[vd.col_key])
vd.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'sample_info': struct {
        npmid: str, 
        sex: str, 
        pid: str, 
        population_code: str, 
        population_name: str, 
        superpopulation_code: str, 
        superpopulation_name: str, 
        data_collections: array<str>
    }
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
----------------------------------------
Entry fields:
    'LA': array<int32>
    'LGT': call
    'LAD': array<int32>
    'LPL': array<int32>
    'RGQ': int32
    'gvcf_info': struct {
        DB: bool, 
        FS: float64, 
        FractionInformativeReads: float64, 
        LOD: float64, 
        MQ: float64, 
        MQRankSum: float64, 
        QD: float64, 
        R2_5P_bias: float64, 
        ReadPosRankSum: float64, 
        SOR: float64
    }
    'AF': array<float64

In [105]:
vd.aggregate_cols(hl.agg.count_where(hl.is_missing(vd.sample_info.npmid))) ## 0

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

0
2025-08-20 12:23:29.209 Hail: INFO: Ordering unsorted dataset with network shuffle
2025-08-20 12:23:31.972 Hail: INFO: Ordering unsorted dataset with network shuffle

In [107]:
vd.filter_cols(vd.sample_info.npmid == 'HG002').cols().show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------+-------------------+-----------------+---------------------+
| s       | sample_info.npmid | sample_info.sex | sample_info.pid     |
+---------+-------------------+-----------------+---------------------+
| str     | str               | str             | str                 |
+---------+-------------------+-----------------+---------------------+
| "HG002" | "HG002"           | NA              | "GIAB_sample_HG002" |
+---------+-------------------+-----------------+---------------------+

+-----------------------------+-----------------------------+
| sample_info.population_code | sample_info.population_name |
+-----------------------------+-----------------------------+
| str                         | str                         |
+-----------------------------+-----------------------------+
| NA                          | NA                          |
+-----------------------------+-----------------------------+

+----------------------------------+------------------------

In [111]:
## add (col-annot) var with sample_info 
vd = vd.annotate_cols(
    sample_info = vd.sample_info.annotate(
        npmid = hl.or_else(vd.sample_info.npmid, vd.s),
        pid   = hl.or_else(vd.sample_info.pid, hl.format('GIAB_sample_%s', vd.s)),
        data_collections = hl.or_else(
            vd.sample_info.data_collections,
            hl.array(['The Genome in a Bottle (GIAB) Consortium'])
        )
    )
)
vd.describe()
vd.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'sample_info': struct {
        npmid: str, 
        sex: str, 
        pid: str, 
        population_code: str, 
        population_name: str, 
        superpopulation_code: str, 
        superpopulation_name: str, 
        data_collections: array<str>
    }
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
----------------------------------------
Entry fields:
    'LA': array<int32>
    'LGT': call
    'LAD': array<int32>
    'LPL': array<int32>
    'RGQ': int32
    'gvcf_info': struct {
        DB: bool, 
        FS: float64, 
        FractionInformativeReads: float64, 
        LOD: float64, 
        MQ: float64, 
        MQRankSum: float64, 
        QD: float64, 
        R2_5P_bias: float64, 
        ReadPosRankSum: float64, 
        SOR: float64
    }
    'AF': array<float64

In [113]:
## create vds from (original) vds ref  + (col-annot) var 
new_vds = hl.vds.VariantDataset(
    reference_data = vds.reference_data,
    variant_data = vd
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [114]:
new_vds.reference_data.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    'ref_block_max_length': int32
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
----------------------------------------
Entry fields:
    'LEN': int32
    'DP': int32
    'GQ': int32
    'ICNT': array<int32>
    'MIN_DP': int32
    'SPL': array<int32>
    'LGT': call
    'LAD': array<int32>
    'END': int32
----------------------------------------
Column key: ['s']
Row key: ['locus']
----------------------------------------

In [115]:
new_vds.variant_data.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'sample_info': struct {
        npmid: str, 
        sex: str, 
        pid: str, 
        population_code: str, 
        population_name: str, 
        superpopulation_code: str, 
        superpopulation_name: str, 
        data_collections: array<str>
    }
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
----------------------------------------
Entry fields:
    'LA': array<int32>
    'LGT': call
    'LAD': array<int32>
    'LPL': array<int32>
    'RGQ': int32
    'gvcf_info': struct {
        DB: bool, 
        FS: float64, 
        FractionInformativeReads: float64, 
        LOD: float64, 
        MQ: float64, 
        MQRankSum: float64, 
        QD: float64, 
        R2_5P_bias: float64, 
        ReadPosRankSum: float64, 
        SOR: float64
    }
    'AF': array<float64

In [116]:
## write to trust staging bucket  
new_vds.write(vds_trust_uri)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2025-08-20 12:43:47.709 Hail: INFO: wrote matrix table with 2887027773 rows and 3205 columns in 5767 partitions to s3://precise-trust/opendata-1000genomes/1000genomes-vds-n3205.vds/reference_data
2025-08-20 12:43:50.143 Hail: INFO: Ordering unsorted dataset with network shuffle
2025-08-20 12:43:50.796 Hail: INFO: Ordering unsorted dataset with network shuffle
2025-08-20 12:47:07.262 Hail: INFO: wrote matrix table with 156228032 rows and 3205 columns in 5767 partitions to s3://precise-trust/opendata-1000genomes/1000genomes-vds-n3205.vds/variant_data